In [135]:
from pathlib import Path
import pymupdf

In [136]:
pdf_path = Path("../data/raw")
list(pdf_path.glob("*.pdf"))

[WindowsPath('../data/raw/microsoft_report.pdf'),
 WindowsPath('../data/raw/test.pdf'),
 WindowsPath('../data/raw/worldbank_report.pdf')]

In [137]:
pdf_path = next(pdf_path.glob("*test*"))

In [138]:
doc = pymupdf.open(pdf_path)

In [139]:
# Création d'identifiant
document_id = pdf_path.stem

In [140]:
document_id

'test'

In [141]:
print(len(doc))
print(doc.metadata)

2
{'format': 'PDF 1.4', 'title': 'Document sans titre', 'author': '', 'subject': '', 'keywords': '', 'creator': '', 'producer': 'Skia/PDF m155 Google Docs Renderer', 'creationDate': '', 'modDate': '', 'trapped': '', 'encryption': None}


In [142]:
pages_document = []

for page_index, page in enumerate(doc):
    page_structure = {
        "document_id": document_id,
        "page": page_index + 1,
        "width": page.rect.width,
        "height": page.rect.height,
        "elements": [] # dimensions de la page (surlignage en front)
    }

    pages_document.append(page_structure)

In [143]:
print(len(pages_document))

for page_structure in pages_document:
    print(page_structure)

2
{'document_id': 'test', 'page': 1, 'width': 596.0, 'height': 842.0, 'elements': []}
{'document_id': 'test', 'page': 2, 'width': 596.0, 'height': 842.0, 'elements': []}


## Extract tables element

In [144]:
for page_index, page in enumerate(doc):
    resultats_tableaux = page.find_tables()

    print(f"Page {page_index + 1} :",len(resultats_tableaux.tables),"tableau(x) détecté(s)")

    for tableau_index, tableau in enumerate(resultats_tableaux.tables):
        print(f"Tableau {tableau_index} :", tableau.row_count, "lignes", tableau.col_count, "colonnes")

        print("Bbox :", list(tableau.bbox))

Page 1 : 1 tableau(x) détecté(s)
Tableau 0 : 20 lignes 3 colonnes
Bbox : [72.5, 238.5, 297.5, 568.5]
Page 2 : 0 tableau(x) détecté(s)


## Add tables to pages

In [145]:
for page_index, page in enumerate(doc):
    table_result = page.find_tables()

    for table_index, table in enumerate(table_result.tables):

        data_table = table.extract()

        if not data_table:
            continue
        
        columns = data_table[0]
        rows = data_table[1:]

        table_element = {
            "type": "table",
            "table_index": table_index,
            "bbox": list(table.bbox),
            "columns": columns,
            "rows": rows
        }

        pages_document[page_index]["elements"].append(table_element)

In [146]:
pages_document

[{'document_id': 'test',
  'page': 1,
  'width': 596.0,
  'height': 842.0,
  'elements': [{'type': 'table',
    'table_index': 0,
    'bbox': [72.5, 238.5, 297.5, 568.5],
    'columns': ['a', 'b', 'c'],
    'rows': [['1', '1', '1'],
     ['2', '2', '2'],
     ['3', '3', '3'],
     ['4', '4', '4'],
     ['5', '5', '5'],
     ['6', '6', '6'],
     ['7', '7', '7'],
     ['8', '8', '8'],
     ['9', '9', '9'],
     ['10', '10', '10'],
     ['11', '11', '11'],
     ['12', '12', '12'],
     ['13', '13', '13'],
     ['14', '14', '14'],
     ['15', '15', '15'],
     ['16', '16', '16'],
     ['17', '17', '17'],
     ['18', '18', '18'],
     ['19', '19', '19']]}]},
 {'document_id': 'test',
  'page': 2,
  'width': 596.0,
  'height': 842.0,
  'elements': []}]

## Extract text element

In [147]:
for page_index, page in enumerate(doc):
    text_blocks = page.get_text("blocks", sort=True)

    table_elements = [element for element in pages_document[page_index]["elements"] if element["type"] == "table"]

    for block in text_blocks:
        print(block)

(272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039, 'TEST \n', 0, 0)
(72.0, 87.3157730102539, 280.3662414550781, 96.2532730102539, 'Qu’est-ce que la génération à enrichissement contextuel ? \n', 1, 0)
(72.0, 110.27476501464844, 505.14263916015625, 119.21226501464844, 'La génération à enrichissement contextuel (RAG) est le processus consistant à optimiser le résultat d’un grand modèle de \n', 2, 0)
(72.0, 124.99349975585938, 509.201171875, 133.93099975585938, 'langage. Elle fait donc appel à une base de connaissances fiable externe aux sources de données utilisées pour l’entraîner \n', 3, 0)
(72.0, 139.71224975585938, 525.177734375, 148.64974975585938, 'avant de générer une réponse. Les grands modèles de langage (LLM) sont entraînés avec d’importants volumes de données et \n', 4, 0)
(72.0, 154.43099975585938, 522.0401611328125, 163.36849975585938, 'utilisent des milliards de paramètres pour générer des résultats originaux pour des tâches telles que répondre à 

In [148]:
for page_index, page in enumerate(doc):
    text_blocks = page.get_text("blocks", sort=True)

    table_elements = [element for element in pages_document[page_index]["elements"] if element["type"] == "table"]

    for block in text_blocks:
        [x0,y0,x1,y1,text,block_number,block_type] = block

        if block_type != 0:
            continue

        text = text.strip()

        if not text:
            continue

        text_bbox = [x0,y0,x1,y1]

        text_rectangle = pymupdf.Rect(text_bbox)

        inside_table = False

        for table_element in table_elements:
            table_rectangle = pymupdf.Rect(table_element["bbox"])

            # Vérification de la présence d'une intersection entre les deux rectangles
            if table_rectangle.intersects(text_rectangle):
                inside_table = True
                break

        if inside_table:
            continue

        text_element = {
            "type": 'text',
            "block_number": block_number,
            "bbox": text_bbox,
            "content": text
        }

        pages_document[page_index]["elements"].append(text_element)

In [149]:
for page in pages_document:
    print(page)

{'document_id': 'test', 'page': 1, 'width': 596.0, 'height': 842.0, 'elements': [{'type': 'table', 'table_index': 0, 'bbox': [72.5, 238.5, 297.5, 568.5], 'columns': ['a', 'b', 'c'], 'rows': [['1', '1', '1'], ['2', '2', '2'], ['3', '3', '3'], ['4', '4', '4'], ['5', '5', '5'], ['6', '6', '6'], ['7', '7', '7'], ['8', '8', '8'], ['9', '9', '9'], ['10', '10', '10'], ['11', '11', '11'], ['12', '12', '12'], ['13', '13', '13'], ['14', '14', '14'], ['15', '15', '15'], ['16', '16', '16'], ['17', '17', '17'], ['18', '18', '18'], ['19', '19', '19']]}, {'type': 'text', 'block_number': 0, 'bbox': [272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039], 'content': 'TEST'}, {'type': 'text', 'block_number': 1, 'bbox': [72.0, 87.3157730102539, 280.3662414550781, 96.2532730102539], 'content': 'Qu’est-ce que la génération à enrichissement contextuel ?'}, {'type': 'text', 'block_number': 2, 'bbox': [72.0, 110.27476501464844, 505.14263916015625, 119.21226501464844], 'content': 'La gén

In [150]:
for page in pages_document:
    print("Page:", page["page"])

    for element in page["elements"]:
        print(element["type"], "|", element["bbox"])

        if element["type"] == "text":
            print(repr(element["content"]))

Page: 1
table | [72.5, 238.5, 297.5, 568.5]
text | [272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039]
'TEST'
text | [72.0, 87.3157730102539, 280.3662414550781, 96.2532730102539]
'Qu’est-ce que la génération à enrichissement contextuel ?'
text | [72.0, 110.27476501464844, 505.14263916015625, 119.21226501464844]
'La génération à enrichissement contextuel (RAG) est le processus consistant à optimiser le résultat d’un grand modèle de'
text | [72.0, 124.99349975585938, 509.201171875, 133.93099975585938]
'langage. Elle fait donc appel à une base de connaissances fiable externe aux sources de données utilisées pour l’entraîner'
text | [72.0, 139.71224975585938, 525.177734375, 148.64974975585938]
'avant de générer une réponse. Les grands modèles de langage (LLM) sont entraînés avec d’importants volumes de données et'
text | [72.0, 154.43099975585938, 522.0401611328125, 163.36849975585938]
'utilisent des milliards de paramètres pour générer des résultats originaux po

## Extract images

In [151]:
for page_index, page in enumerate(doc):
    page_images = page.get_images(full=True)

    for image_index, image_info in enumerate(page_images):
        xref = image_info[0] # identifiant interne
        width = image_info[2]
        height = image_info[3]

        image_positions = page.get_image_rects(xref) # récupération des positions de l'image sur la page

        for occurrence_index, position in enumerate(image_positions):
            image_id = (f"{document_id}_page_{page_index + 1}_image_{image_index}_occurrence_{occurrence_index}") # occurrence: combien de fois une même image est affiché sur la page

            image_element = {
                "type": "image",
                "image_id": image_id,
                "image_index": image_index,
                "xref": xref,
                "width": width,
                "height": height,
                "bbox": list(position)
            }

            pages_document[page_index]["elements"].append(image_element)

In [152]:
for page in pages_document:
    print("\nPage:", page["page"])

    for element in page["elements"]:
        print(element["type"], "|", element["bbox"])

        if element["type"] == "image":
            print("ID:",element["image_id"])

            print("Size", element["width"],"x", element["height"])


Page: 1
table | [72.5, 238.5, 297.5, 568.5]
text | [272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039]
text | [72.0, 87.3157730102539, 280.3662414550781, 96.2532730102539]
text | [72.0, 110.27476501464844, 505.14263916015625, 119.21226501464844]
text | [72.0, 124.99349975585938, 509.201171875, 133.93099975585938]
text | [72.0, 139.71224975585938, 525.177734375, 148.64974975585938]
text | [72.0, 154.43099975585938, 522.0401611328125, 163.36849975585938]
text | [72.0, 169.14976501464844, 506.937744140625, 178.08726501464844]
text | [72.0, 183.86851501464844, 498.52020263671875, 192.80601501464844]
text | [72.0, 198.58726501464844, 509.1426086425781, 207.52476501464844]
text | [72.0, 209.68490600585938, 180.18663024902344, 223.09115600585938]

Page: 2
image | [73.5, 66.9749755859375, 215.25, 208.7249755859375]
ID: test_page_2_image_0_occurrence_0
Size 200 x 200
image | [73.5, 278.3123779296875, 525.0, 560.3123779296875]
ID: test_page_2_image_1_occurrence_0
Size

### Recording each images

image_id : occurence visible sur la page

asset_id : fichier image stocké

storage_path = emplacement du fichier

In [153]:
# Création banque d'images

image_bank_root = Path("image_bank")

image_bank_root.mkdir(exist_ok=True)

image_bank = []
saved_assets = set() # évite d’enregistrer plusieurs fois la même image si elle possède plusieurs occurrences

In [154]:
for page_index, page in enumerate(pages_document):
    page_number = page["page"]

    page_folder = image_bank_root/document_id/f"page_{page_number}"

    page_folder.mkdir(parents=True,exist_ok=True)

    for element in page["elements"]:
        if element["type"] != "image":
            continue

        asset_id = f"{document_id}_page_{page_number}_image_{element['image_index']}"

        element["asset_id"] = asset_id

        if asset_id in saved_assets:
            continue

        extracted_image = doc.extract_image(element["xref"])

        extension = extracted_image["ext"]

        file_name = f"image_{element['image_index']}.{extension}"

        file_path = page_folder/file_name

        file_path.write_bytes(extracted_image["image"])

        storage_path = file_path.as_posix() # pour stocker par exemple : image_bank/test/page_2/image_0.jpeg, plus portable pour une application

        element["storage_path"] = storage_path

        image_record = {
            "asset_id": asset_id,
            "document_id": document_id,
            "page": page_number,
            "image_index": element["image_index"],
            "xref": element["xref"],
            "width": element["width"],
            "height": element["height"],
            "extension": extension,
            "storage_path": storage_path
        }

        image_bank.append(image_record)

        saved_assets.add(asset_id)

In [155]:
print("Number of stored images:", len(image_bank))

for image_record in image_bank:
    print(image_record)

Number of stored images: 2
{'asset_id': 'test_page_2_image_0', 'document_id': 'test', 'page': 2, 'image_index': 0, 'xref': 9, 'width': 200, 'height': 200, 'extension': 'jpeg', 'storage_path': 'image_bank/test/page_2/image_0.jpeg'}
{'asset_id': 'test_page_2_image_1', 'document_id': 'test', 'page': 2, 'image_index': 1, 'xref': 11, 'width': 855, 'height': 535, 'extension': 'png', 'storage_path': 'image_bank/test/page_2/image_1.png'}


In [156]:
for element in pages_document:
    print(element)

{'document_id': 'test', 'page': 1, 'width': 596.0, 'height': 842.0, 'elements': [{'type': 'table', 'table_index': 0, 'bbox': [72.5, 238.5, 297.5, 568.5], 'columns': ['a', 'b', 'c'], 'rows': [['1', '1', '1'], ['2', '2', '2'], ['3', '3', '3'], ['4', '4', '4'], ['5', '5', '5'], ['6', '6', '6'], ['7', '7', '7'], ['8', '8', '8'], ['9', '9', '9'], ['10', '10', '10'], ['11', '11', '11'], ['12', '12', '12'], ['13', '13', '13'], ['14', '14', '14'], ['15', '15', '15'], ['16', '16', '16'], ['17', '17', '17'], ['18', '18', '18'], ['19', '19', '19']]}, {'type': 'text', 'block_number': 0, 'bbox': [272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039], 'content': 'TEST'}, {'type': 'text', 'block_number': 1, 'bbox': [72.0, 87.3157730102539, 280.3662414550781, 96.2532730102539], 'content': 'Qu’est-ce que la génération à enrichissement contextuel ?'}, {'type': 'text', 'block_number': 2, 'bbox': [72.0, 110.27476501464844, 505.14263916015625, 119.21226501464844], 'content': 'La gén

In [157]:
for element in pages_document:
    print(element.keys())

dict_keys(['document_id', 'page', 'width', 'height', 'elements'])
dict_keys(['document_id', 'page', 'width', 'height', 'elements'])


In [158]:
for page in pages_document:
    for element in page["elements"]:
        if element["type"] == "image":
            print(element["image_id"],"->",element["asset_id"],"->",element.get("storage_path"))

test_page_2_image_0_occurrence_0 -> test_page_2_image_0 -> image_bank/test/page_2/image_0.jpeg
test_page_2_image_1_occurrence_0 -> test_page_2_image_1 -> image_bank/test/page_2/image_1.png


Le tableau a été ajouté avant le texte car l'extraction a été faite en premier. On doit donc mettre la liste dans l'ordre.

### Triage selon ordre d'apparition

In [159]:
for page in pages_document:
    page["elements"] = sorted(
        page["elements"],
        key=lambda element: (
            element["bbox"][1], # triage avec y0
            element["bbox"][0]  # et si plusieurs y0 sont égaux, triage avec x0
        )
    )

In [160]:
for page_data in pages_document:
    print(
        "\nPage:",
        page_data["page"]
    )

    for element in page_data["elements"]:
        print(
            element["type"],
            "| y0:",
            round(element["bbox"][1], 2),
            "| x0:",
            round(element["bbox"][0], 2)
        )


Page: 1
text | y0: 50.26 | x0: 272.09
text | y0: 87.32 | x0: 72.0
text | y0: 110.27 | x0: 72.0
text | y0: 124.99 | x0: 72.0
text | y0: 139.71 | x0: 72.0
text | y0: 154.43 | x0: 72.0
text | y0: 169.15 | x0: 72.0
text | y0: 183.87 | x0: 72.0
text | y0: 198.59 | x0: 72.0
text | y0: 209.68 | x0: 72.0
table | y0: 238.5 | x0: 72.5

Page: 2
image | y0: 66.97 | x0: 73.5
image | y0: 278.31 | x0: 73.5


# Classification des images

In [161]:
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] != "image":
            continue

        storage_path = element.get("storage_path") # .get() : évite KeyError si la clé n'existe pas, retourne None

        print(
            "Page :", page_data["page"],
            "| Image ID :", element["image_id"],
            "| Path :", storage_path
        )

Page : 2 | Image ID : test_page_2_image_0_occurrence_0 | Path : image_bank/test/page_2/image_0.jpeg
Page : 2 | Image ID : test_page_2_image_1_occurrence_0 | Path : image_bank/test/page_2/image_1.png


## CLIP

In [162]:
from transformers import pipeline

In [163]:
image_classifier = pipeline(
    task="zero-shot-image-classification",
    model="openai/clip-vit-base-patch32",
    device=-1
)

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 22716.34it/s]


In [164]:
# Définition des catégories
candidate_labels = [
    "logo",
    "chart",
    "diagram",
    "scanned document",
    "photograph"
]

In [165]:
for i in image_bank:
    print(i)

{'asset_id': 'test_page_2_image_0', 'document_id': 'test', 'page': 2, 'image_index': 0, 'xref': 9, 'width': 200, 'height': 200, 'extension': 'jpeg', 'storage_path': 'image_bank/test/page_2/image_0.jpeg'}
{'asset_id': 'test_page_2_image_1', 'document_id': 'test', 'page': 2, 'image_index': 1, 'xref': 11, 'width': 855, 'height': 535, 'extension': 'png', 'storage_path': 'image_bank/test/page_2/image_1.png'}


In [166]:
# Test
first_image_path = image_bank[0]["storage_path"]
print(first_image_path)

image_bank/test/page_2/image_0.jpeg


In [167]:
# Classification
classification_results = image_classifier(first_image_path,candidate_labels=candidate_labels)

for result in classification_results:
    print(result["label"],":",round(result["score"],3))

logo : 0.911
diagram : 0.054
scanned document : 0.03
chart : 0.005
photograph : 0.001


In [168]:
best_result = classification_results[0]
print(best_result["label"])
print("Confiance:", round(best_result["score"],3))

logo
Confiance: 0.911


In [169]:
# Pour toutes les images avec enregistrement dans chaque dictionnaire d'image_positions

for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] != "image":
            continue

        classification_results = image_classifier(element["storage_path"],candidate_labels=candidate_labels)

        best_result = classification_results[0]

        # Sauvegarde dans le dictionnaire existant
        element["image_category"] = best_result["label"]
        element["classification_score"] = float(best_result["score"]) 

        print(
            element["image_id"], "|Category :", element["image_category"], "|Score :", round(element["classification_score"],3)
        )

test_page_2_image_0_occurrence_0 |Category : logo |Score : 0.911
test_page_2_image_1_occurrence_0 |Category : chart |Score : 0.949


In [170]:
for page_data in pages_document:
    for element in page_data["elements"]:
        if element["type"] == "image":
            print(element["image_id"],"->",element["image_category"],"->",round(element["classification_score"], 3))

test_page_2_image_0_occurrence_0 -> logo -> 0.911
test_page_2_image_1_occurrence_0 -> chart -> 0.949


### Selection des graphiques

In [171]:
chart_element = None

for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] == "image" and element["image_category"] == "chart":
            chart_element = element
            break # Arrêt de la boucle une fois le graphique trouvé

    if chart_element is not None:
        break # Arrêt de la boucle des pages

In [172]:
if chart_element is None:
    print("No chart detected")
else:
    print("Chart detected :", chart_element["image_id"])
    print("Path :", chart_element["storage_path"])

Chart detected : test_page_2_image_1_occurrence_0
Path : image_bank/test/page_2/image_1.png


### DePlot

In [173]:
from transformers import Pix2StructProcessor,Pix2StructForConditionalGeneration

In [174]:
# Modèle
deplot_model_name = "google/deplot"

# Processeur (prépare l'image et la consigne)
deplot_processor = Pix2StructProcessor.from_pretrained('google/deplot')

# Chargement du modèle
deplot_model =  Pix2StructForConditionalGeneration.from_pretrained(deplot_model_name)

# mode inférence, sans entraînement
deplot_model.eval()

Loading weights: 100%|██████████| 285/285 [00:00<00:00, 353.77it/s]


Pix2StructForConditionalGeneration(
  (encoder): Pix2StructVisionModel(
    (embeddings): Pix2StructVisionEmbeddings(
      (patch_projection): Linear(in_features=768, out_features=768, bias=True)
      (row_embedder): Embedding(4096, 768)
      (column_embedder): Embedding(4096, 768)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): Pix2StructVisionEncoder(
      (layer): ModuleList(
        (0-11): 12 x Pix2StructVisionLayer(
          (attention): Pix2StructVisionAttention(
            (query): Linear(in_features=768, out_features=768, bias=False)
            (key): Linear(in_features=768, out_features=768, bias=False)
            (value): Linear(in_features=768, out_features=768, bias=False)
            (output): Linear(in_features=768, out_features=768, bias=False)
          )
          (mlp): Pix2StructVisionMlp(
            (wi_0): Linear(in_features=768, out_features=2048, bias=False)
            (wi_1): Linear(in_features=768, out_features=2048, bias=False)
 

In [175]:
from PIL import Image

chart_image = Image.open(chart_element["storage_path"]).convert("RGB")

In [176]:
model_inputs = deplot_processor(
    images=chart_image,
    text="Generate underlying data table of the figure below:", return_tensors="pt"
)

In [177]:
import torch

with torch.inference_mode(): # Le modèle est utilisé que pour la prédiction, pas de préparations de calculs pour l'entraînement ce qui économise de la mémoire
    predictions = deplot_model.generate(**model_inputs, max_new_tokens=512) # ** transmission au modèle de toutes les données par deplot_processor

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [178]:
raw_chart_table = deplot_processor.decode(predictions[0], skip_special_tokens=True) # Sélection de la première prédiction, transforme les identifiants numériques en texte puis retire les marqueurs technique du modèle

In [179]:
predictions[0]

tensor([    0, 36757,   789, 24138, 10658,   273,   290,   274,   279,   274,
          279,    27,  7560,   741,  5614,   414,   287, 11491,   287,  5614,
         9071,   301,   705,   273,   301,  1887,  1930, 14227,  5741,  1171,
          687,   284,  1887,  4769,   389,   933, 37297,   288,  1023,  8872,
         7195,   275,    27, 12277,   810,   889,  5973,   275,   273,   279,
          278,   277,   279,  1171,   273,   279,   278,   277,   292,    27,
         1749, 10631,   372,   890,   284,  1887,   741,  5614,   414,   287,
          789, 16756, 49370,   287,   789, 20090,   676,    27,   279,   278,
          277,   279,   789,   273,   277,   277,   278,   789,   273,   294,
          290,    27,   279,   278,   277,   285,   789,   273,   277,   294,
          290,   789,   273,   277,   295,   290,    27,   279,   278,   277,
          289,   789,   273,   279,   289,   277,   789,   273,   279,   279,
          289,    27,   279,   278,   277,   290,   789,   273, 

In [180]:
print(raw_chart_table)

TITLE | Graphique 5.2.2<0x0A>Élèves possèdant un télephone intelligent à l'école des Bois-Francs,<0x0A>selon le genre, 2012 à 2019<0x0A>Nombre d'élèves | Garçons | Filles<0x0A>2012 | 110 | 85<0x0A>2013 | 185 | 175<0x0A>2014 | 241 | 224<0x0A>2015 | 286 | 296<0x0A>2016 | 305 | 280<0x0A>2017 | 309 | 315<0x0A>2018 | 314 | 305<0x0A>2019 | 314 | 320


In [181]:
chart_element["deplot_raw"] = raw_chart_table

In [182]:
chart_element

{'type': 'image',
 'image_id': 'test_page_2_image_1_occurrence_0',
 'image_index': 1,
 'xref': 11,
 'width': 855,
 'height': 535,
 'bbox': [73.5, 278.3123779296875, 525.0, 560.3123779296875],
 'asset_id': 'test_page_2_image_1',
 'storage_path': 'image_bank/test/page_2/image_1.png',
 'image_category': 'chart',
 'classification_score': 0.9489662051200867,
 'deplot_raw': "TITLE | Graphique 5.2.2<0x0A>Élèves possèdant un télephone intelligent à l'école des Bois-Francs,<0x0A>selon le genre, 2012 à 2019<0x0A>Nombre d'élèves | Garçons | Filles<0x0A>2012 | 110 | 85<0x0A>2013 | 185 | 175<0x0A>2014 | 241 | 224<0x0A>2015 | 286 | 296<0x0A>2016 | 305 | 280<0x0A>2017 | 309 | 315<0x0A>2018 | 314 | 305<0x0A>2019 | 314 | 320"}

### Plusieurs graphiques par page

In [183]:
chart_element = None

for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] != "image":
            continue

        if element["image_category"] != "chart":
            continue

        print("Analyzing chart :", element["image_id"])

        chart_image = Image.open(
            element["storage_path"]
        ).convert("RGB")

        model_inputs = deplot_processor(
            images=chart_image,
            text="Generate underlying data table of the figure below:",
            return_tensors="pt"
        )

        with torch.inference_mode():
            predictions = deplot_model.generate(
                **model_inputs,
                max_new_tokens=512
            )

        raw_chart_table = deplot_processor.decode(
            predictions[0], # un graphique par boucle donc [0]
            skip_special_tokens=True
        )

        element["deplot_raw"] = raw_chart_table

Analyzing chart : test_page_2_image_1_occurrence_0


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [184]:
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] == "image" and element["image_category"] == "chart":
            print("\nChart :", element["image_id"])
            print("DePlot result :", element["deplot_raw"])


Chart : test_page_2_image_1_occurrence_0
DePlot result : TITLE | Graphique 5.2.2<0x0A>Élèves possèdant un télephone intelligent à l'école des Bois-Francs,<0x0A>selon le genre, 2012 à 2019<0x0A>Nombre d'élèves | Garçons | Filles<0x0A>2012 | 110 | 85<0x0A>2013 | 185 | 175<0x0A>2014 | 241 | 224<0x0A>2015 | 286 | 296<0x0A>2016 | 305 | 280<0x0A>2017 | 309 | 315<0x0A>2018 | 314 | 305<0x0A>2019 | 314 | 320


<0x0A> : pointe un retour à la ligne

### Remplacement de <0x0A> par retour à la ligne

In [185]:
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] != "image":
            continue

        if element["image_category"] != "chart":
            continue

        clean_chart_table = element["deplot_raw"].replace( "<0x0A>","\n")

        element["deplot_clean"] = clean_chart_table

In [186]:
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] == "image" and element["image_category"] == "chart":
            print("\nChart :", element["image_id"])
            print(element["deplot_clean"])


Chart : test_page_2_image_1_occurrence_0
TITLE | Graphique 5.2.2
Élèves possèdant un télephone intelligent à l'école des Bois-Francs,
selon le genre, 2012 à 2019
Nombre d'élèves | Garçons | Filles
2012 | 110 | 85
2013 | 185 | 175
2014 | 241 | 224
2015 | 286 | 296
2016 | 305 | 280
2017 | 309 | 315
2018 | 314 | 305
2019 | 314 | 320


In [187]:
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] != "image":
            continue

        if element["image_category"] != "chart":
            continue

        chart_lines = [
            line.strip() # retire les espaces inutiles au début et à la fin de chaque ligne
            for line in element["deplot_clean"].splitlines() # découpe le texte à chaque retour à la ligne
            if line.strip() # ignore les lignes vides
        ]

        element["deplot_lines"] = chart_lines

In [188]:
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] == "image" and element["image_category"] == "chart":

            print("\nChart :", element["image_id"])

            for line_index, line in enumerate(element["deplot_lines"]):
                print(line_index, ":", repr(line))


Chart : test_page_2_image_1_occurrence_0
0 : 'TITLE | Graphique 5.2.2'
1 : "Élèves possèdant un télephone intelligent à l'école des Bois-Francs,"
2 : 'selon le genre, 2012 à 2019'
3 : "Nombre d'élèves | Garçons | Filles"
4 : '2012 | 110 | 85'
5 : '2013 | 185 | 175'
6 : '2014 | 241 | 224'
7 : '2015 | 286 | 296'
8 : '2016 | 305 | 280'
9 : '2017 | 309 | 315'
10 : '2018 | 314 | 305'
11 : '2019 | 314 | 320'


Lignes du titre : 1 séparateur |

En-tête : 2 séparateurs

Lignes données 2 séparateurs

### En-tête

In [189]:
# Première ligne ayant au moins 2 séparateurs (en_tête)

for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] != "image":
            continue

        if element["image_category"] != "chart":
            continue

        header_index = None

        for line_index, line in enumerate(element["deplot_lines"]):

            if line.count("|") >=2:
                header_index = line_index
                break

        element["deplot_header_index"] = header_index


In [190]:
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] == "image" and element["image_category"] == "chart":

            header_index = element["deplot_header_index"]

            print("Header index :", header_index)

            if header_index is not None:
                print(
                    "Header line :",
                    element["deplot_lines"][header_index]
                )

Header index : 3
Header line : Nombre d'élèves | Garçons | Filles


### Extraction des colonnes

In [191]:
# Noms des colonnes
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] != "image":
            continue

        if element["image_category"] != "chart":
            continue

        header_index = element["deplot_header_index"]

        if header_index is None:
            continue

        header_line = element["deplot_lines"][header_index]

        chart_columns = [value.strip() for value in header_line.split("|")]

        element["chart_columns"] = chart_columns

In [192]:
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] == "image" and element["image_category"] == "chart":
            print(
                "Chart :",
                element["image_id"]
            )
            print(
                "Columns :",
                element["chart_columns"]
            )

Chart : test_page_2_image_1_occurrence_0
Columns : ["Nombre d'élèves", 'Garçons', 'Filles']


### Extraction des lignes

In [193]:
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] != "image":
            continue

        if element["image_category"] != "chart":
            continue

        header_index = element["deplot_header_index"]
        chart_columns = element["chart_columns"]

        chart_rows = []

        for line in element["deplot_lines"][header_index + 1:]: # Ligne qui suit l'en-tête

            values = [
                value.strip()
                for value in line.split("|") # Liste de valeurs
            ]

            if len(values) != len(chart_columns):
                continue

            structured_row = dict(
                zip(chart_columns, values) # Associe les colonnes aux valeurs sous forme de dictionnaire
            )

            chart_rows.append(structured_row)

        element["chart_rows"] = chart_rows

In [194]:
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] == "image" and element["image_category"] == "chart":

            print("Number of rows :", len(element["chart_rows"]))

            for row in element["chart_rows"]:
                print(row)

Number of rows : 8
{"Nombre d'élèves": '2012', 'Garçons': '110', 'Filles': '85'}
{"Nombre d'élèves": '2013', 'Garçons': '185', 'Filles': '175'}
{"Nombre d'élèves": '2014', 'Garçons': '241', 'Filles': '224'}
{"Nombre d'élèves": '2015', 'Garçons': '286', 'Filles': '296'}
{"Nombre d'élèves": '2016', 'Garçons': '305', 'Filles': '280'}
{"Nombre d'élèves": '2017', 'Garçons': '309', 'Filles': '315'}
{"Nombre d'élèves": '2018', 'Garçons': '314', 'Filles': '305'}
{"Nombre d'élèves": '2019', 'Garçons': '314', 'Filles': '320'}


### Extraction du titre du tableau

In [195]:
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] != "image":
            continue

        if element["image_category"] != "chart":
            continue

        header_index = element["deplot_header_index"]

        title_lines = element["deplot_lines"][:header_index] # Les lignes avant header_index

        if title_lines and title_lines[0].startswith("TITLE |"): # vérifie que la liste n'est pas vide puis que la première ligne commence par le marqueur produit par DePlot
            title_lines[0] = title_lines[0].split("|",1)[1].strip() # "|",1 : split au premier | si présence de plusieurs | . [1] : recupère le 2ème élément car 1er élément -> TITLE

        chart_title = " ".join(title_lines)   

        element["chart_title"] = chart_title      

In [196]:
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] == "image" and element["image_category"] == "chart":
            print("Chart title :", element["chart_title"])

Chart title : Graphique 5.2.2 Élèves possèdant un télephone intelligent à l'école des Bois-Francs, selon le genre, 2012 à 2019


### Regroupement des informations extraites du graphique dans une seule clé

In [197]:
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] != "image":
            continue

        if element["image_category"] != "chart":
            continue

        chart_data = {
            "title": element["chart_title"],
            "columns": element["chart_columns"],
            "rows": element["chart_rows"]
        }

        element["chart_data"] = chart_data

In [198]:
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] == "image" and element["image_category"] == "chart":
            print("Chart :", element["image_id"])
            print(element["chart_data"])

Chart : test_page_2_image_1_occurrence_0
{'title': "Graphique 5.2.2 Élèves possèdant un télephone intelligent à l'école des Bois-Francs, selon le genre, 2012 à 2019", 'columns': ["Nombre d'élèves", 'Garçons', 'Filles'], 'rows': [{"Nombre d'élèves": '2012', 'Garçons': '110', 'Filles': '85'}, {"Nombre d'élèves": '2013', 'Garçons': '185', 'Filles': '175'}, {"Nombre d'élèves": '2014', 'Garçons': '241', 'Filles': '224'}, {"Nombre d'élèves": '2015', 'Garçons': '286', 'Filles': '296'}, {"Nombre d'élèves": '2016', 'Garçons': '305', 'Filles': '280'}, {"Nombre d'élèves": '2017', 'Garçons': '309', 'Filles': '315'}, {"Nombre d'élèves": '2018', 'Garçons': '314', 'Filles': '305'}, {"Nombre d'élèves": '2019', 'Garçons': '314', 'Filles': '320'}]}


# Conversion en Markdown

## Tableau

In [199]:
def table_to_markdown(table_element):

    columns = table_element["columns"]
    rows = table_element["rows"]

    markdown_lines = []

    header_line = "| " + " | ".join(columns) + " |"
    markdown_lines.append(header_line)

    separator_values = ["---"] * len(columns)
    separator_line = "| " + " | ".join(separator_values) + " |"
    markdown_lines.append(separator_line)

    for row in rows:

        row_values = [
            str(value) if value is not None else "" # si la valeur existe, elle est convertie avec str(), si la cellule contient None, elle devient une chaîne vide
            for value in row
        ]

        row_line = "| " + " | ".join(row_values)+ " |"

        markdown_lines.append(row_line)

    return "\n".join(markdown_lines)

In [200]:
test_table = None

for page_data in pages_document:
    for element in page_data["elements"]:
        if element["type"] == "table":
            test_table = element
            break

    if test_table is not None:
        break

In [201]:
table_markdown = table_to_markdown(table_element)

print(table_markdown)

| a | b | c |
| --- | --- | --- |
| 1 | 1 | 1 |
| 2 | 2 | 2 |
| 3 | 3 | 3 |
| 4 | 4 | 4 |
| 5 | 5 | 5 |
| 6 | 6 | 6 |
| 7 | 7 | 7 |
| 8 | 8 | 8 |
| 9 | 9 | 9 |
| 10 | 10 | 10 |
| 11 | 11 | 11 |
| 12 | 12 | 12 |
| 13 | 13 | 13 |
| 14 | 14 | 14 |
| 15 | 15 | 15 |
| 16 | 16 | 16 |
| 17 | 17 | 17 |
| 18 | 18 | 18 |
| 19 | 19 | 19 |


## Texte

In [202]:
def text_to_markdown(text_element):

    content = text_element["content"]

    return content

In [203]:
text_element = None

for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] == "text":
            text_element = element
            break

    if text_element is not None:
        break

In [204]:
text_markdown = text_to_markdown(text_element)

print(text_markdown)

TEST


## Images

In [205]:
def image_to_markdown(image_element):

    image_id = image_element["image_id"]
    image_category = image_element["image_category"]

    image_markdown = (
        f"![{image_category}]"
        f"(asset://{image_id})" # pour obtenir par exemple: ![logo](asset://test_page_2_image_0_occurence_0)
    )

    return image_markdown

In [206]:
test_image = None

for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] == "image":
            test_image = element
            break

    if test_image is not None:
        break

In [207]:
image_markdown = image_to_markdown(test_image)

print(image_markdown)

![logo](asset://test_page_2_image_0_occurrence_0)


In [208]:
def chart_to_markdown(image_element):

    chart_data = image_element["chart_data"]

    title = chart_data["title"]
    columns = chart_data["columns"]
    structured_rows = chart_data["rows"]

    rows = []

    for structured_row in structured_rows:

        row = [
            structured_row.get(column, "")
            for column in columns
        ]

        rows.append(row)

    return rows

In [209]:
test_chart = None

for page_data in pages_document:
    for element in page_data["elements"]:

        if (
            element["type"] == "image"
            and element["image_category"] == "chart"
        ):
            test_chart = element
            break

    if test_chart is not None:
        break

In [210]:
chart_rows = chart_to_markdown(test_chart)

print(chart_rows[0])

['2012', '110', '85']


Maintenant que les lignes du graphique ont le bon format, nous pouvons les envoyer à table_to_markdown()

table_to_markdown(table_element) attend un dictionnaire avec columns et rows

In [211]:
def chart_to_markdown(image_element):

    chart_data = image_element["chart_data"]

    title = chart_data["title"]
    columns = chart_data["columns"]
    structured_rows = chart_data["rows"]

    rows = []

    for structured_row in structured_rows:

        row = [
            structured_row.get(column, "")
            for column in columns
        ]

        rows.append(row)

    temporary_table = {
        "columns": columns,
        "rows": rows
    }

    chart_table_markdown = table_to_markdown(temporary_table)

    chart_markdown = (
        f"### {title}\n\n"
        f"{chart_table_markdown}"
    )

    return chart_markdown

In [212]:
chart_markdown = chart_to_markdown(test_chart)

print(chart_markdown)

### Graphique 5.2.2 Élèves possèdant un télephone intelligent à l'école des Bois-Francs, selon le genre, 2012 à 2019

| Nombre d'élèves | Garçons | Filles |
| --- | --- | --- |
| 2012 | 110 | 85 |
| 2013 | 185 | 175 |
| 2014 | 241 | 224 |
| 2015 | 286 | 296 |
| 2016 | 305 | 280 |
| 2017 | 309 | 315 |
| 2018 | 314 | 305 |
| 2019 | 314 | 320 |


### Modification de image_to_markdown() afin qu’elle ajoute automatiquement les données DePlot lorsqu’il s’agit d’un graphique.

In [213]:
def image_to_markdown(image_element):

    asset_id = image_element["asset_id"]
    image_category = image_element["image_category"]

    image_reference = (
        f"![{image_category}]"
        f"(asset://{asset_id})"
    )

    if (image_category == "chart" and "chart_data" in image_element):
        chart_markdown = chart_to_markdown(image_element)

        return (
            f"{image_reference}\n\n"
            f"{chart_markdown}"
        )

    return image_reference

In [238]:
for page_data in pages_document:
    for element in page_data["elements"]:

        if element["type"] == "image":
            result = image_to_markdown(element)

            print(result.splitlines()[0])

![logo](asset://test_page_2_image_0)
![chart](asset://test_page_2_image_1)


In [214]:
image_markdown = image_to_markdown(test_chart)

print(image_markdown)

![chart](asset://test_page_2_image_1)

### Graphique 5.2.2 Élèves possèdant un télephone intelligent à l'école des Bois-Francs, selon le genre, 2012 à 2019

| Nombre d'élèves | Garçons | Filles |
| --- | --- | --- |
| 2012 | 110 | 85 |
| 2013 | 185 | 175 |
| 2014 | 241 | 224 |
| 2015 | 286 | 296 |
| 2016 | 305 | 280 |
| 2017 | 309 | 315 |
| 2018 | 314 | 305 |
| 2019 | 314 | 320 |


## Fonction choisissant automatiquement la bonne conversion selon le type de l’élément

In [215]:
def element_to_markdown(element):

    element_type = element["type"]

    if element_type == "text":
        return text_to_markdown(element)

    if element_type == "table":
        return table_to_markdown(element)

    if element_type == "image":
        return image_to_markdown(element)

    return "" # Si un nouveau type inconnu apparaît, la fonction retourne une chaîne vide au lieu de provoquer une erreur

In [216]:
for page_data in pages_document:

    print("\nPAGE :", page_data["page"])

    for element in page_data["elements"]:

        element_markdown = element_to_markdown(element)

        print("\n---")
        print(element_markdown)


PAGE : 1

---
TEST

---
Qu’est-ce que la génération à enrichissement contextuel ?

---
La génération à enrichissement contextuel (RAG) est le processus consistant à optimiser le résultat d’un grand modèle de

---
langage. Elle fait donc appel à une base de connaissances fiable externe aux sources de données utilisées pour l’entraîner

---
avant de générer une réponse. Les grands modèles de langage (LLM) sont entraînés avec d’importants volumes de données et

---
utilisent des milliards de paramètres pour générer des résultats originaux pour des tâches telles que répondre à des questions,

---
traduire des langues et compléter des phrases. La RAG étend les capacités déjà très puissantes des LLM à des domaines

---
spécifiques ou à la base de connaissances interne d’une organisation, le tout sans qu’il soit nécessaire de réentraîner le

---
modèle. Il s’agit d’une approche économique pour améliorer les résultats du LLM et qu’ils restent cohérents, précis et utiles

---
dans de nombreux 

## Obtention d'une seule chaîne Markdown par page

In [217]:
page_markdowns = [] # Liste chaine markdown par page

for page_data in pages_document:

    markdown_parts = [] # Liste d'éléments convertis

    for element in page_data["elements"]:

        element_markdown = element_to_markdown(element)

        if element_markdown:
            markdown_parts.append(element_markdown)

    page_markdown = "\n\n".join(markdown_parts) # 2 retours à la ligne pour bien séparer les markdown

    page_markdowns.append(page_markdown)

In [218]:
print("Number of pages :", len(page_markdowns))

for page_index, page_markdown in enumerate(page_markdowns,start=1):
    print("\n==========")
    print("PAGE", page_index)
    print("==========")
    print(page_markdown)

Number of pages : 2

PAGE 1
TEST

Qu’est-ce que la génération à enrichissement contextuel ?

La génération à enrichissement contextuel (RAG) est le processus consistant à optimiser le résultat d’un grand modèle de

langage. Elle fait donc appel à une base de connaissances fiable externe aux sources de données utilisées pour l’entraîner

avant de générer une réponse. Les grands modèles de langage (LLM) sont entraînés avec d’importants volumes de données et

utilisent des milliards de paramètres pour générer des résultats originaux pour des tâches telles que répondre à des questions,

traduire des langues et compléter des phrases. La RAG étend les capacités déjà très puissantes des LLM à des domaines

spécifiques ou à la base de connaissances interne d’une organisation, le tout sans qu’il soit nécessaire de réentraîner le

modèle. Il s’agit d’une approche économique pour améliorer les résultats du LLM et qu’ils restent cohérents, précis et utiles

dans de nombreux contextes.

| a | b | c

# Chunks avec leur contenu et leurs métadonnées

In [219]:
chunks = []

for page_index, page_data in enumerate(pages_document):

    # Éléments de la page actuelle
    elements = page_data["elements"]

    # Types présents sur la page, sans doublon
    element_types = list(
        dict.fromkeys(
            element["type"]
            for element in elements
        )
    )

    # Identifiants des fichiers présents dans la banque d'images
    image_ids = list(
        dict.fromkeys(
            element["asset_id"]
            for element in elements
            if element["type"] == "image"
        )
    )

    # Catégories d'images détectées par CLIP
    image_categories = list(
        dict.fromkeys(
            element["image_category"]
            for element in elements
            if element["type"] == "image"
        )
    )

    # Éléments permettant de localiser les sources sur la page
    source_elements = []

    for element_index, element in enumerate(elements):

        source_element = {
            "element_id": (
                f"{document_id}"
                f"_page_{page_data['page']}"
                f"_element_{element_index}"
            ),
            "type": element["type"],
            "bbox": element["bbox"]
        }

        # Informations supplémentaires pour les images
        if element["type"] == "image":
            source_element["asset_id"] = element["asset_id"]
            source_element["image_category"] = element["image_category"]

        source_elements.append(source_element)

    # Création du chunk de la page

    chunk_id = f"{document_id}_page_{page_data['page']}"

    chunk = {
        "chunk_id": chunk_id,

        # Contenu utilisé pour la recherche sémantique
        "page_content": page_markdowns[page_index],

        # Informations simples conservées avec le vecteur
        "metadata": {
            "chunk_id": chunk_id,
            "file_name": pdf_path.name,
            "document_id": document_id,
            "page": page_data["page"],
            "page_width": page_data["width"],
            "page_height": page_data["height"],
            "element_types": element_types,
            "image_categories": image_categories,
            "has_images": len(image_ids) > 0,
            "image_ids": image_ids
        },

        # Positions des éléments pour le futur surlignage
        "source_elements": source_elements
    }

    chunks.append(chunk)

In [220]:
print("Number of chunks :", len(chunks))

for chunk in chunks:

    print("\nChunk ID :", chunk["chunk_id"])
    print("Metadata :", chunk["metadata"])

    print("Source elements :")

    for source_element in chunk["source_elements"]:
        print(source_element)

Number of chunks : 2

Chunk ID : test_page_1
Metadata : {'chunk_id': 'test_page_1', 'file_name': 'test.pdf', 'document_id': 'test', 'page': 1, 'page_width': 596.0, 'page_height': 842.0, 'element_types': ['text', 'table'], 'image_categories': [], 'has_images': False, 'image_ids': []}
Source elements :
{'element_id': 'test_page_1_element_0', 'type': 'text', 'bbox': [272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039]}
{'element_id': 'test_page_1_element_1', 'type': 'text', 'bbox': [72.0, 87.3157730102539, 280.3662414550781, 96.2532730102539]}
{'element_id': 'test_page_1_element_2', 'type': 'text', 'bbox': [72.0, 110.27476501464844, 505.14263916015625, 119.21226501464844]}
{'element_id': 'test_page_1_element_3', 'type': 'text', 'bbox': [72.0, 124.99349975585938, 509.201171875, 133.93099975585938]}
{'element_id': 'test_page_1_element_4', 'type': 'text', 'bbox': [72.0, 139.71224975585938, 525.177734375, 148.64974975585938]}
{'element_id': 'test_page_1_element_5', '

# Embedding

In [221]:
texts_to_embed = [chunk["page_content"]for chunk in chunks]

In [222]:
print("Number of texts to embed :", len(texts_to_embed))

for text_index, text in enumerate(texts_to_embed):
    print("Text", text_index, "| Length :", len(text))

Number of texts to embed : 2
Text 0 | Length : 1290
Text 1 | Length : 420


In [223]:
import os 
from dotenv import load_dotenv
load_dotenv(override=True)

os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')

In [224]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-ada-002")

In [225]:
embeddings = embedding_model.embed_documents(texts_to_embed)

In [226]:
print("Number of embeddings :", len(embeddings))

for embedding_index, embedding in enumerate(embeddings):
    print("Chunk :", chunks[embedding_index]["chunk_id"], "| Dimensions :", len(embedding))

Number of embeddings : 2
Chunk : test_page_1 | Dimensions : 1536
Chunk : test_page_2 | Dimensions : 1536


# Chromadb

In [227]:
import chromadb

chroma_client = chromadb.PersistentClient(path="chroma_db") # Les données seront conservées sur le disque dans le fichier chroma_db

In [228]:
# Création de la collection (table)

collection = chroma_client.get_or_create_collection(name="pdf_pages")

In [229]:
for chunk in chunks:
    print(chunk)

{'chunk_id': 'test_page_1', 'page_content': 'TEST\n\nQu’est-ce que la génération à enrichissement contextuel ?\n\nLa génération à enrichissement contextuel (RAG) est le processus consistant à optimiser le résultat d’un grand modèle de\n\nlangage. Elle fait donc appel à une base de connaissances fiable externe aux sources de données utilisées pour l’entraîner\n\navant de générer une réponse. Les grands modèles de langage (LLM) sont entraînés avec d’importants volumes de données et\n\nutilisent des milliards de paramètres pour générer des résultats originaux pour des tâches telles que répondre à des questions,\n\ntraduire des langues et compléter des phrases. La RAG étend les capacités déjà très puissantes des LLM à des domaines\n\nspécifiques ou à la base de connaissances interne d’une organisation, le tout sans qu’il soit nécessaire de réentraîner le\n\nmodèle. Il s’agit d’une approche économique pour améliorer les résultats du LLM et qu’ils restent cohérents, précis et utiles\n\ndans 

In [230]:
# Identifiants qui seront utilisés dans Chroma

chunk_ids = [chunk["chunk_id"] for chunk in chunks]

In [231]:
print(chunk_ids)

['test_page_1', 'test_page_2']


In [232]:
# Contenus Markdown associés aux identifiants

chunk_documents = [chunk["page_content"] for chunk in chunks]

Nous avons 3 listes alignées:

- chunk_ids
- chunk_documents
- embeddings

Exemple pour la page 1:

- chunk_ids[0]
- chunk_documents[0]
- embeddings[0]

In [233]:
# Enregistrement des identifiants, contenus Markdown et embeddings dans Chroma

collection.upsert(ids=chunk_ids, documents=chunk_documents, embeddings=embeddings)

In [234]:
print("Number of records in Chroma :",collection.count())

Number of records in Chroma : 2


In [235]:
from langchain_chroma import Chroma

vector_store = Chroma(client=chroma_client, collection_name="pdf_pages", embedding_function=embedding_model)

# Retriever

In [236]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [251]:
question = "Combien de filles possédaient un téléphone intelligent en 2019?"

retrieved_documents = retriever.invoke(question)

print("Number of retrieved documents :", len(retrieved_documents))

for rank, document in enumerate(retrieved_documents,start=1):
    print("\nRank :", rank)
    print("Document ID :", document.id)
    print("Content :")
    print(document.page_content)

Number of retrieved documents : 2

Rank : 1
Document ID : test_page_2
Content :
![logo](asset://test_page_2_image_0)

![chart](asset://test_page_2_image_1)

### Graphique 5.2.2 Élèves possèdant un télephone intelligent à l'école des Bois-Francs, selon le genre, 2012 à 2019

| Nombre d'élèves | Garçons | Filles |
| --- | --- | --- |
| 2012 | 110 | 85 |
| 2013 | 185 | 175 |
| 2014 | 241 | 224 |
| 2015 | 286 | 296 |
| 2016 | 305 | 280 |
| 2017 | 309 | 315 |
| 2018 | 314 | 305 |
| 2019 | 314 | 320 |

Rank : 2
Document ID : test_page_1
Content :
TEST

Qu’est-ce que la génération à enrichissement contextuel ?

La génération à enrichissement contextuel (RAG) est le processus consistant à optimiser le résultat d’un grand modèle de

langage. Elle fait donc appel à une base de connaissances fiable externe aux sources de données utilisées pour l’entraîner

avant de générer une réponse. Les grands modèles de langage (LLM) sont entraînés avec d’importants volumes de données et

utilisent des millia

# LLM

In [243]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-mini")

In [244]:
# Index chunk complet
chunks_by_id = {
    chunk["chunk_id"]: chunk
    for chunk in chunks
}

In [245]:
# Préparation source
context_parts = []

for document in retrieved_documents:

    source_chunk = chunks_by_id[
        document.id
    ]

    page_number = source_chunk["metadata"]["page"]
    file_name = source_chunk["metadata"]["file_name"]

    context_part = (
        f"[Source: {file_name}, page {page_number}]\n"
        f"{document.page_content}"
    )

    context_parts.append(context_part)

In [246]:
context = "\n\n---\n\n".join(
    context_parts
)

In [247]:
print(context)

[Source: test.pdf, page 2]
![logo](asset://test_page_2_image_0)

![chart](asset://test_page_2_image_1)

### Graphique 5.2.2 Élèves possèdant un télephone intelligent à l'école des Bois-Francs, selon le genre, 2012 à 2019

| Nombre d'élèves | Garçons | Filles |
| --- | --- | --- |
| 2012 | 110 | 85 |
| 2013 | 185 | 175 |
| 2014 | 241 | 224 |
| 2015 | 286 | 296 |
| 2016 | 305 | 280 |
| 2017 | 309 | 315 |
| 2018 | 314 | 305 |
| 2019 | 314 | 320 |

---

[Source: test.pdf, page 1]
TEST

Qu’est-ce que la génération à enrichissement contextuel ?

La génération à enrichissement contextuel (RAG) est le processus consistant à optimiser le résultat d’un grand modèle de

langage. Elle fait donc appel à une base de connaissances fiable externe aux sources de données utilisées pour l’entraîner

avant de générer une réponse. Les grands modèles de langage (LLM) sont entraînés avec d’importants volumes de données et

utilisent des milliards de paramètres pour générer des résultats originaux pour des tâ

In [248]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Tu es un assistant qui répond à partir de documents fournis.

Utilise uniquement les informations présentes dans les sources.
Si les sources ne permettent pas de répondre, indique que l'information n'a pas été trouvée.
Indique le nom du fichier et le numéro de page utilisés.
Réponds en français de manière claire et concise.
""".strip()
        ),
        (
            "human",
            """
Question :
{question}

Sources :
{context}
""".strip()
        )
    ]
)

In [250]:
chain = prompt_template | llm

In [252]:
question = "Combien de filles possédaient un téléphone intelligent en 2019?"

response = chain.invoke(
    {
        "question": question,
        "context": context
    }
)

In [253]:
print(response.content)

320 filles possédaient un téléphone intelligent en 2019 (test.pdf, page 2).


In [257]:
def rag_answer(question: str):

    # Recherche des pages pertinentes
    retrieved_documents = retriever.invoke(
        question
    )

    # Construction du contexte
    context_parts = []

    for document in retrieved_documents:

        source_chunk = chunks_by_id[
            document.id
        ]

        file_name = source_chunk["metadata"]["file_name"]
        page_number = source_chunk["metadata"]["page"]

        context_part = (
            f"[Source: {file_name}, page {page_number}]\n"
            f"{document.page_content}"
        )

        context_parts.append(context_part)

    context = "\n\n---\n\n".join(
        context_parts
    )

    # Génération de la réponse
    response = chain.invoke(
        {
            "question": question,
            "context": context
        }
    )

    return response.content

In [258]:
rag_answer("Combien de filles possédaient un téléphone intelligent en 2019?")

'En 2019, 320 filles possédaient un téléphone intelligent (test.pdf, page 2).'

In [259]:
rag_answer("Qu'est ce qu'un rag?")

"D'après le document, un RAG (génération à enrichissement contextuel) est le processus visant à optimiser le résultat d’un grand modèle de langage en faisant appel à une base de connaissances externe et fiable, distincte des données utilisées pour l’entraîner, avant de générer une réponse. Cette approche permet d’étendre les capacités des LLM à des domaines spécifiques ou à la base de connaissances interne d’une organisation sans réentraîner le modèle, de manière économique, pour obtenir des réponses plus cohérentes, précises et utiles.  \n\nSource : test.pdf, page 1."

In [260]:
rag_answer("Si j'ai a à 10 à combien est égal b et c?")

'D’après le tableau, lorsque a = 10, b = 10 et c = 10.  \nSource : test.pdf, page 1.'